# Using SVG Visualizations

In [1]:
from svg_snip.Composer import Composer
from svg_snip.Composer import Group, comment

import svg_snip.Elements as e2d

def x(x=0, y=0, **kwargs):
    return f"""\
<line x1="{x-5}" y1="{y-5}" x2="{x+5}" y2="{y+5}" stroke="white"/>
<line x1="{x-5}" y1="{y+5}" x2="{x+5}" y2="{y-5}" stroke="white"/>"""

svg = Composer((100,100))
svg.add(e2d.rect, x=0, y=0, width=100, height=100)
svg.add(x, x=50, y=50)

svg_g = Group()
svg_g.add(comment, text="This is the svg_g Group")
svg_g.add(e2d.circle, cx=30, cy=20, r=5, stroke='yellow', fill='red')

svg.add(svg_g)

svg.display(debug=True)

In [2]:
# Here is how the basic drawing works: functions that return text!
e2d.circle?
e2d.circle(cx=30, cy=20, r=5, stroke='yellow', fill='red')

'<circle cx="30.00" cy="20.00" r="5.00" fill="red" stroke="yellow" />'

Signature: e2d.circle(**kwargs)
Docstring:
Generate SVG code for the <circle> element.

Parameters:
- cx: (float): The x-coordinate of the center of the circle.
- cy: (float): The y-coordinate of the center of the circle.
- r: (float): The radius of the circle.
- fill: (str): The fill color. Examples: 'currentColor' keyword, 'red', #FF0000, rgb(255,0,0) ...
- fill_opacity: (float): The opacity of the shape's fill. A value between 0 (completely transparent) and 1 (completely opaque).
- fill_rule: (str): Defines the inside of the shape to dertermine the fill region. Possible values are 'nonzero' and 'evenodd'.
- pattern: (str): URL referencing a <pattern> element to be used as the fill.
- gradient: (str): URL referencing a <linearGradient> or <radialGradient> element to be used as the fill.
- stroke: (str): The stroke color of the outline. Examples: 'currentColor' keyword, 'red', #FF0000, rgb(255,0,0) ...
- stroke_width: (float): The width of the outline.
- stroke_opacity: (float): The o

# How to interact with simple UI elements

Jupyter is decently fast with updating short HTML fragments.
You loose most time because running the python kernel in the backend and sending resulting text to the frontend.

For debugging and scientific visualizations, this is quite fast enough.

In [3]:
import ipywidgets as w
from IPython.display import clear_output, display

s = w.IntSlider(value=0, min=0, max=360, step=2, description='Angle')
out = w.Output()

def update(c):
    with out:
        clear_output(wait=True)
        svg = Composer((100,100))
        svg.add(e2d.rect, x=0, y=0, width=100, height=100)
        g = Group()
        g.add(e2d.rect, x=20, y=20, width=30, height=30, fill='blue')
        svg.add(g, transform=f'rotate({c["new"]} 50 50)')
        svg.display()

s.observe(update, names='value')
display(s, out)
update({"new": s.value})

IntSlider(value=0, description='Angle', max=360, step=2)

Output()

# Using 3D Visualizations

SVG does not suport 3D rendering. But it is a vector graphic that is easily exported and used in papers, for example.

This is easily fast enough to support interactively tuning parameters and rotating the camera for simple 3D geometry.

Note that the backface removal is based on the normal. So shaded faces work only for simple, convex geometry.

In [4]:
from svg_snip.Jupyter import CanvasWithOverlay
import svg_snip.Elements as e2d
import svg_snip.Elements3D as e3d
import ipywidgets as w

import numpy as np

from ProjectiveGeometry23.central_projection import ProjectionMatrix
from ProjectiveGeometry23.homography import rotation_x, rotation_z

P_lookat = ProjectionMatrix.perspective_look_at(
    eye=np.array([0, 0, 250]),
    center=np.array([0, 0, 0]),
    image_size=(500, 500),
    fovy_rad=0.7
)

# Color sliders
r = w.IntSlider(value=0, min=0, max=255, description='R')
g = w.IntSlider(value=120, min=0, max=255, description='G')
b = w.IntSlider(value=255, min=0, max=255, description='B')

vis = CanvasWithOverlay(500, 500)

ax = az = 1

def draw():
    svg = Composer((vis.w, vis.h))

    svg.add(
        e3d.cube,
        min=[-50,-50,-50],
        max=[50,50,50],
        fill=f'rgb({r.value},{g.value},{b.value})',
        stroke='black'
    )

    T = rotation_x(ax) @ rotation_z(az)
    vis.html_overlay.value = svg.render(P=P_lookat.P @ T)


def handle_draw(vis):
    global ax
    global az

    if vis.mouse_state.clicked:
        az += vis.mouse_state.dx * -0.01
        ax += vis.mouse_state.dy * 0.01

    draw()


vis.handle_draw = handle_draw

# Redraw when sliders change
for s in (r, g, b):
    s.observe(lambda _: draw(), names='value')

display(r, g, b)
vis.display()
draw()

IntSlider(value=0, description='R', max=255)

IntSlider(value=120, description='G', max=255)

IntSlider(value=255, description='B', max=255)

In [5]:
print(vis.html_overlay.value)

<svg  width="500" height="500" viewBox="0 0 500 500" xmlns="http://www.w3.org/2000/svg">
  <g>
    <!-- Shaded Cube -->
    <polygon points="187.44,269.71 49.21,395.92 86.66,169.76 202.88,1.55" fill="rgb(0,120,255)" stroke="black" style="filter:brightness(72%)" />
    <polygon points="187.44,269.71 474.98,360.52 286.69,443.44 49.21,395.92" fill="rgb(0,120,255)" stroke="black" style="filter:brightness(77%)" />
    <polygon points="187.44,269.71 202.88,1.55 429.0,119.91 474.98,360.52" fill="rgb(0,120,255)" stroke="black" style="filter:brightness(85%)" />
  </g>
</svg>


In [6]:
with open('example_3D_cube.svg', 'w') as file:
    file.write(vis.html_overlay.value)